## Basics

In [20]:
#import relevant libraries
import os
from scipy import stats
import numpy as np
import scipy as sp
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm
import seaborn as sns
import dabest
import NLCLIMB
import NLMATH
import itertools
from datetime import datetime
date = datetime.today().strftime('%Y%m%d')
from statistics import mean
from textwrap import wrap
import dabest
import plotly.express as px 
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from plotly.graph_objects import Layout
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import osar 

#NOTE: SUPPRESSES WARNINGS!

import warnings
warnings.simplefilter(action="ignore", category=RuntimeWarning)
warnings.simplefilter(action="ignore", category=UserWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)

print(osar.__version__)

0.23.9


In [21]:
officecomp = "C:\\Users\\Star\\"
labcomp = "C:\\Users\\User\\"
computer2 = "C:\\Users\\lnico\\"
homecomp = "D:\\"
specifiedpath = homecomp

filedirectory_OSAR = "ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\Data Compilation\\osar_compiled\\2025collection\\"
filedirectory_Falling = "ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\Data Compilation\\Falling_New\\Compilation with delta\\2025deltagcollection\\"
filesavedir = specifiedpath + "ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\Data Compilation\\Totalosarfalling\\images\\"
filedate = "20260126"

newfile2 = specifiedpath + "ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\Data Compilation\\Falling_New\\Compilation with delta\\2025fallingtoosarcomp\\"
files2 = os.listdir(newfile2)

In [22]:
#for fonts only
import matplotlib.pyplot as plt
from matplotlib import font_manager

font_dirs = [specifiedpath + "\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\Programs\\"]  # The path to the custom font file.
font_files = font_manager.findSystemFonts(fontpaths=font_dirs)

for font_file in font_files:
    font_manager.fontManager.addfont(font_file)
    prop = font_manager.FontProperties(fname=font_file)

availablefonts = [f.name for f in matplotlib.font_manager.fontManager.ttflist]
plt.rcParams["font.family"] = "Inter"

## Function

In [23]:
def reading_OSAR_compiled_files(df, responder):
    df['genotypeandresponder'] = df['MBON'] + "_" + responder
    
    return df

def forestplot_multiplot(sorteddf, df1, parameter):
    lst = []
    names =[]
    for n in sorteddf['MBON']:
        tempdf1 = df1[(df1['MBON'] == n)].reset_index(drop=True)
        dbfeature = dabest.load(tempdf1, idx=("Sibling", "Offspring"),x="status", y= parameter)
        lst.append(dbfeature)
        names.append(n)
    return lst, names



In [24]:
newfile2 = specifiedpath + filedirectory_Falling  # from 2025deltagcollection
files2 = os.listdir(newfile2)

## Single files for Falling

In [25]:
totalfile = pd.DataFrame()

for n in files2:
    openfile = pd.read_csv(specifiedpath + filedirectory_Falling + n)
    nameofmbon = n.split(" ")[0] 
    openfile['responder'] = n.split(" x ")[1].split("_")[0]
    totalfile = pd.concat([totalfile, openfile], axis = 0).reset_index(drop=True)

lstofallvariables = []
for n in totalfile.columns.to_list()[1:-1]:
    lstofallvariables.append(n.split("_")[0])
lstnamings= list(set(lstofallvariables))

totalfile['genotypeandresponder'] = totalfile["MBON"] + "_" + totalfile['responder']   #has bootstraps

#only delta-g
onlymeans = totalfile.loc[:, ~totalfile.columns.str.endswith('_bootstrap')].drop_duplicates().reset_index(drop=True)

mbononly=onlymeans[~onlymeans['MBON'].isin(['R58', 'Th-Gal4', "R76B09", "VT999036"])]

# to generate excel file only
fallingexcel = mbononly.copy()
fallingexcel.columns = fallingexcel.columns.str.replace('_deltag', '_Δg')
fallingexcel = fallingexcel.rename(columns={'speed_Δg': 'Speed_Δg', 
                                            'bspeed_Δg': 'Bout speed_Δg', 
                                            'pausepos_Δg': 'Pause position_Δg', 
                                            'bout_Δg': 'Number of Bouts_Δg', 
                                            'meanbout_Δg': "Duration of bouts_Δg",
                      'straightindex_Δg': 'Straightness Index_Δg', 
                      'height_Δg': 'Average height climbed_Δg', 
                      'maxvelocity_Δg': 'Max velocity_Δg', 
                      "fallnumber_meandiff": "Fall number_ΔΔ",
                      'log2bspeed_hedgesg':'Log2 Bout Speed ratio_Hg', 
                     'log2speed_hedgesg': 'Log2 Speed ratio_Hg', 
                     'boutnumber_index_hedgesg': 'Bout Number index_Hg', 
                     'boutduration_ratio_hedgesg': "Bout Duration ratio_Hg",
       'maxvelocity_ratio_hedgesg': 'Max Velocity ratio_Hg'}).reset_index(drop=True)

fallingexcel.to_excel(specifiedpath+ "ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\Data Compilation\\2025 Complete raw values osar falling\\Falling_" + date + ".xlsx", index = False)

                     

## Single files for OSAR from already generated compiled sheet

In [17]:
responder = "ACR"
#from dabest.forest_plot import forest_plot
filedate = "20260423"
compiled = specifiedpath + "ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\Data Compilation\\osar_compiled\\"+ filedate + "_" + responder + "_totalcompilation.csv"
dfcompiled= pd.read_csv(compiled, index_col=0)
dfcompiled.replace([np.inf, -np.inf], np.nan, inplace=True)
dfcompiled.rename(columns={'driver': 'MBON'}, inplace = True)
dfcompiled=dfcompiled[~dfcompiled['MBON'].isin(['R58', 'Th-Gal4', 'VT999036', 'R76B09'])]

In [18]:
# Define the four conditions
conditions = {
    'Eighth': dfcompiled['light_intensity'] == 'Eighth',
    'Quarter': dfcompiled['light_intensity'] == 'Quarter',
    'Half': dfcompiled['light_intensity'] == 'Half',
    'Full': dfcompiled['light_intensity'] == 'Full',
    'Half_and_Full': (dfcompiled['light_intensity'] == 'Half') | (dfcompiled['light_intensity'] == 'Full')
}
# conditions = {    
#     'Full': dfcompiled['light_intensity'] == 'Full',
    
# }

for condition_name, condition_filter in conditions.items():
    # Filter data based on condition
    df1 = dfcompiled[condition_filter].loc[:,['MBON','light_intensity','status','pi_smoothed_Pattern 01', 
                                               'log2_speed_ratio_Pattern 01','log2_pace_ratio_Pattern 01', 'speed_ratio_Pattern 01', 'pace_ratio_Pattern 01',
                                               'light_attraction_index_Pattern 01', 'bout_index_Pattern 01', 'bout_duration_ratio_Pattern 01',
                                               'max_velocity_ratio_Pattern 01']].reset_index(drop=True)

    sorteddf = pd.DataFrame()
    
    for n in df1['MBON'].unique().tolist():
        dbfeature = dabest.load(df1[(df1['MBON'] == n)].reset_index(drop=True), idx=("Sibling", "Offspring"),x="status", y="pi_smoothed_Pattern 01")
        tempdf = pd.DataFrame()
        tempdf['MBON'] = [n]
        tempdf['difference']= [float(dbfeature.hedges_g.statistical_tests.difference)]
        sorteddf = pd.concat([sorteddf, tempdf], axis =0)
    
    sorteddf = sorteddf.sort_values(by=['difference']).reset_index(drop=True)

    # Generate excel files
    parameters= ["pi_smoothed_Pattern 01", "log2_speed_ratio_Pattern 01", "log2_pace_ratio_Pattern 01", "light_attraction_index_Pattern 01", 
                 "bout_index_Pattern 01", "bout_duration_ratio_Pattern 01", "max_velocity_ratio_Pattern 01",'speed_ratio_Pattern 01', 'pace_ratio_Pattern 01',
  ]

    parameters_rename= ["PI_Hg_OSAR", "Log2 Speed ratio_Hg_OSAR", "Log2 Bout Speed ratio_Hg_OSAR", "Light Attraction Index_Hg_OSAR",
                        'Bout Number Index_Hg_OSAR', 'Bout Duration ratio_Hg_OSAR', 'Max Velocity ratio_Hg_OSAR', "Speed ratio_Hg_OSAR", "Bout Speed ratio_Hg_OSAR",
   ]
        
    result_df = pd.DataFrame({'MBON': sorteddf['MBON'].unique()})
    
    for parameter, newname in zip(parameters, parameters_rename):
        differences = []
        
        for n in result_df['MBON']:
            tempdf1 = df1[(df1['MBON'] == n)].reset_index(drop=True)
            dbfeature = dabest.load(tempdf1, idx=("Sibling", "Offspring"), x="status", y=parameter)
            differences.append(dbfeature.hedges_g.statistical_tests.difference[0])
        
        # Add as new column
        result_df[newname] = differences
    
    # Save with condition name in filename
    result_df.to_excel(specifiedpath + "ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\Data Compilation\\2025 Complete raw values osar falling\\OSAR_" + responder + "-" + condition_name + ".xlsx", index=False)
    
    print(f"Saved: OSAR_{responder}_{condition_name}.xlsx")

Saved: OSAR_ACR_Eighth.xlsx
Saved: OSAR_ACR_Quarter.xlsx
Saved: OSAR_ACR_Half.xlsx
Saved: OSAR_ACR_Full.xlsx
Saved: OSAR_ACR_Half_and_Full.xlsx


## OSAR-Falling Combination

In [7]:
#pre processed fallin
root = specifiedpath + "ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\Data Compilation\\2025 Complete raw values osar falling\\"
filefalling = root + "Falling_20260215.xlsx"
df_falling = pd.read_excel(filefalling)

cols = df_falling.columns.tolist()
cols[1:-2] = [col + '_Falling' for col in cols[1:-2]]
df_falling.columns = cols

df_falling = df_falling.drop(columns= ["MBON"])

lightintensities = ['Eighth', 'Quarter','Half', 'Full', 'Half_and_Full']
root = specifiedpath + "ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\Data Compilation\\2025 Complete raw values osar falling\\"

for light in lightintensities:
    filepathACR = root + "OSAR_ACR-"+ light +".xlsx"
    filepathCR2 = root + "OSAR_Chrimson2-" + light + ".xlsx"

    dffileACR= reading_OSAR_compiled_files(pd.read_excel(filepathACR), "ACR")
    dffileCR2= reading_OSAR_compiled_files(pd.read_excel(filepathCR2),'Chrimson2')
    dfosartotal= pd.concat([dffileACR, dffileCR2], axis = 0).reset_index(drop=True)    

    dfmix = pd.DataFrame()
    dfmix = df_falling.merge(dfosartotal, on="genotypeandresponder", how="inner")
    dfmix = dfmix.sort_values(by = ["genotypeandresponder"])

    totalmetriccomparison = dfmix[~dfmix['MBON'].isin(['R58', 'Th-Gal4'])]

    totalmetriccomparison.to_excel(specifiedpath + "ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\Data Compilation\\2025 Complete raw values osar falling\\Totalcomparisonofallmetrics-" + light +".xlsx", index = False)
    print(f"Saved: {light}.xlsx")

Saved: Eighth.xlsx
Saved: Quarter.xlsx
Saved: Half.xlsx
Saved: Full.xlsx
Saved: Half_and_Full.xlsx
